In [46]:
from DrissionPage import ChromiumPage
import pandas as pd
import time
import os

def fast_crawl_with_resume(base_url, start_page, end_page):
    page = ChromiumPage()
    output_file = 'readtoolead_full_dataset.csv'
    
    # Đọc dữ liệu cũ
    processed_links = set()
    if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
        try:
            # Đọc toàn bộ file để kiểm tra cột
            df_check = pd.read_csv(output_file, nrows=0) 
            if 'Source' in df_check.columns:
                df_existing = pd.read_csv(output_file, usecols=['Source'])
                processed_links = set(df_existing['Source'].dropna().astype(str).unique())
                print(f" Đã tìm thấy file cũ. Hiện có {len(processed_links)} link. ")
            else:
                print(" File cũ thiếu cột 'Source'.")
        except Exception as e:
            print(f" Lỗi đọc file: {e}")

    try:
        for p_num in range(start_page, end_page + 1):
            list_url = f"{base_url}/page/{p_num}/" if p_num > 1 else base_url
            print(f"\n--- Đang quét trang {p_num} ---")
            
            page.get(list_url)

            all_elements = page.eles('css:.tdb_module_loop_2 .entry-title a')
            
            # Lấy link và chuẩn hóa dấu gạch chéo
            all_links = [a.link.rstrip('/') for a in all_elements]
            
            # Lọc trùng dựa trên set() đã chuẩn hóa
            processed_normalized = [p.rstrip('/') for p in processed_links]
            new_links = [l for l in list(dict.fromkeys(all_links)) if l not in processed_normalized]
            
            print(f"Tổng : {len(all_links)} | Mới: {len(new_links)} | Cũ: {len(all_links) - len(new_links)}")

            if not new_links:
                continue

            for link in new_links:
                try:
                    processed_links.add(link)
                    print(f" Đang xử lý bài: {link}")
                    page.get(link)
                    
                    # JS Click đồng loạt
                    page.run_js("document.querySelectorAll('a.bg-showmore-plg-link').forEach(btn => { if(btn.innerText.includes('Hiển thị')) btn.click(); });")
                    time.sleep(1.2)

                    title = page.ele('tag:h1').text if page.ele('tag:h1') else "No Title"
                    blocks = page.eles('.bg-margin-for-link')
                    
                    current_article_data = []
                    for block in blocks:
                        vi_div = block.ele('css:div[id^="bg-showmore-hidden"]')
                        vi_text = vi_div.text if vi_div else ""
                        
                        # Duyệt ngược vét cạn tìm English
                        en_text = ""
                        prev_node = block.prev()
                        while prev_node:
                            p_text = prev_node.text.strip()
                            if len(p_text) > 10 and "tiếng Việt" not in p_text:
                                en_text = p_text
                                break
                            prev_node = prev_node.prev()
                        
                        if en_text and vi_text:
                            current_article_data.append({
                                'Article_Title': title,
                                'English': en_text,
                                'Vietnamese': vi_text,
                                'Source': link 
                            })
                    
                    if current_article_data:
                        df_new = pd.DataFrame(current_article_data)
                        # Ghi header nếu file chưa tồn tại
                        header_needed = not os.path.exists(output_file) or os.path.getsize(output_file) == 0
                        df_new.to_csv(output_file, mode='a', index=False, header=header_needed, encoding='utf-8-sig')
                        print(f" Đã thêm {len(current_article_data)} cặp câu.")

                except Exception as e:
                    print(f" Lỗi: {e}")
                    continue

    finally:
        page.quit()
        if os.path.exists(output_file):
            final_df = pd.read_csv(output_file)
            print(f"\n TỔNG KẾT: File hiện có {len(final_df)} dòng dữ liệu.")

if __name__ == "__main__":
    fast_crawl_with_resume("https://readtoolead.com", start_page=4, end_page=10) 

 Đã tìm thấy file cũ. Hiện có 195 link. 

--- Đang quét trang 4 ---
Tổng : 10 | Mới: 0 | Cũ: 10

--- Đang quét trang 5 ---
Tổng : 10 | Mới: 0 | Cũ: 10

--- Đang quét trang 6 ---
Tổng : 10 | Mới: 0 | Cũ: 10

--- Đang quét trang 7 ---
Tổng : 10 | Mới: 0 | Cũ: 10

--- Đang quét trang 8 ---
Tổng : 10 | Mới: 0 | Cũ: 10

--- Đang quét trang 9 ---
Tổng : 10 | Mới: 0 | Cũ: 10

--- Đang quét trang 10 ---
Tổng : 10 | Mới: 0 | Cũ: 10

 TỔNG KẾT: File hiện có 2666 dòng dữ liệu.
